# Chapter 3 Exercises - Classification

Refactored exercises from Hands-On Machine Learning, now using modular code.

In [1]:
# Setup for Colab compatibility
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/scaling-octo-broccoli'
    sys.path.insert(0, PROJECT_PATH)
    %cd {PROJECT_PATH}
else:
    # Local - go to project root
    from pathlib import Path
    PROJECT_PATH = str(Path.cwd().parent)
    sys.path.insert(0, PROJECT_PATH)

In [2]:
# Import project modules
from src.data import MNISTLoader, TitanicLoader
from src.preprocessing import ImageAugmentor, TitanicPreprocessor
from src.models import ClassifierFactory
from src.training import Trainer, HyperparameterSearcher

print("Imports successful!")

Imports successful!


## Exercise 1: MNIST KNN Classifier (>97% accuracy)

Build a classifier for MNIST that achieves over 97% accuracy using KNeighborsClassifier with grid search for hyperparameters.

In [5]:
# Load MNIST data
# Update path based on where you store the data
mnist_path = "/home/gon/Documents/ML/HandsOnPythorch/scaling-octo-broccoli/data/raw/mnist_784.csv.zip"  # or full path for Colab

loader = MNISTLoader(mnist_path)
print(f"Dataset shape: {loader.shape}")

# Stratified split
X_train, X_test, y_train, y_test = loader.stratified_split(test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Dataset shape: (70000, 785)
Training set: (56000, 784)
Test set: (14000, 784)


In [ ]:
# Hyperparameter search
from sklearn.neighbors import KNeighborsClassifier

param_grid = {
    "n_neighbors": [3, 5, 7, 9, 10],
    "weights": ["uniform", "distance"]
}

searcher = HyperparameterSearcher(
    model=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=4,
    scoring="accuracy"
)

print("Starting grid search (this may take a while)...")
searcher.grid_search(X_train, y_train)

print(f"\nBest parameters: {searcher.best_params}")
print(f"Best CV score: {searcher.best_score:.4f}")

In [ ]:
# Evaluate on test set
trainer = Trainer(searcher.best_estimator)
trainer._is_fitted = True  # Already fitted by grid search

results = trainer.evaluate(X_test, y_test)
print(f"Test set accuracy: {results['accuracy']:.4f}")

## Exercise 2: Data Augmentation

Augment the training set by creating shifted copies of each image (left, right, up, down by 1 pixel).

In [ ]:
# Create augmented dataset
augmentor = ImageAugmentor(image_shape=(28, 28))

X_train_aug, y_train_aug = augmentor.augment_with_shifts(
    X_train, y_train,
    shifts=[(1, 0), (-1, 0), (0, 1), (0, -1)],  # 4 directions
    shuffle=True,
    random_state=42
)

print(f"Original training size: {len(X_train)}")
print(f"Augmented training size: {len(X_train_aug)}")
print(f"Augmentation factor: {augmentor.augment_factor()}x")

In [ ]:
# Train with best params on augmented data
model = ClassifierFactory.create("knn", params=searcher.best_params)
trainer = Trainer(model)

print("Training on augmented data...")
trainer.fit(X_train_aug, y_train_aug)

# Evaluate on original test set
results = trainer.evaluate(X_test, y_test)
print(f"Test accuracy with augmentation: {results['accuracy']:.4f}")

## Exercise 3: Titanic Classification

Train classifiers on the Titanic dataset and compare performance.

In [ ]:
# Load Titanic data
titanic_loader = TitanicLoader("data/raw")
titanic_loader.download()  # Downloads if not present

train_df, test_df = titanic_loader.load()
print(f"Training samples: {len(train_df)}")
print(f"Features: {titanic_loader.feature_names}")

In [ ]:
# Preprocess data
X, y = titanic_loader.get_features_and_labels()

preprocessor = TitanicPreprocessor(
    numerical_features=["Age", "SibSp", "Parch", "Fare"],
    categorical_features=["Pclass", "Sex", "Embarked"]
)

X_processed = preprocessor.fit_transform(X)
print(f"Processed features shape: {X_processed.shape}")
print(f"Feature names: {preprocessor.get_feature_names()[:5]}...")  # First 5

In [ ]:
# Compare classifiers
results = {}

for model_name in ["random_forest", "svm", "knn"]:
    model = ClassifierFactory.create(model_name)
    trainer = Trainer(model)
    
    cv_results = trainer.cross_validate(X_processed, y, cv=10)
    results[model_name] = cv_results
    
    print(f"{model_name:15} - Mean: {cv_results['mean']:.4f} (+/- {cv_results['std']:.4f})")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt

names = list(results.keys())
means = [r['mean'] for r in results.values()]
stds = [r['std'] for r in results.values()]

plt.figure(figsize=(10, 6))
plt.bar(names, means, yerr=stds, capsize=5, color=['#2ecc71', '#3498db', '#e74c3c'])
plt.ylabel('Cross-Validation Accuracy')
plt.title('Classifier Comparison on Titanic Dataset')
plt.ylim(0.7, 0.9)
for i, (m, s) in enumerate(zip(means, stds)):
    plt.text(i, m + s + 0.01, f'{m:.3f}', ha='center')
plt.tight_layout()
plt.show()

## Summary

Results from Chapter 3 exercises:

1. **MNIST KNN**: Best params `n_neighbors=3, weights='distance'` achieved ~97.3% accuracy
2. **Data Augmentation**: 5x training data with pixel shifts (improvement varies)
3. **Titanic**: SVM slightly outperforms Random Forest and KNN (~82% vs ~81%)